# 🚀 Taller: AI-HPC Convergence
### CPU vs GPU — Benchmark en tiempo real
**CINF102 · Fundamentos de Cómputo de Alto Desempeño · 2026-10**

---

## ⚙️ Antes de empezar
1. Ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución**
2. Selecciona **GPU T4** (gratuita)
3. Haz clic en **Guardar** y luego en **Conectar**

> **Objetivo:** Medir empíricamente el speedup de GPU sobre CPU, y verificar los límites
> teóricos usando la **Ley de Amdahl**. Sin instalar nada — todo corre en la nube.


---
## 📋 Sección 1 — Verificar entorno GPU
Primero confirmamos que Colab nos asignó una GPU real.

In [ ]:
# ─── Verificar GPU con nvidia-smi ───────────────────────────────────────────
!nvidia-smi

In [ ]:
import torch

gpu_disponible = torch.cuda.is_available()
print(f"¿GPU disponible?  {'✅ SÍ' if gpu_disponible else '❌ NO — revisa el tipo de entorno'}")

if gpu_disponible:
    nombre_gpu = torch.cuda.get_device_name(0)
    memoria_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Modelo GPU:       {nombre_gpu}")
    print(f"Memoria VRAM:     {memoria_gb:.1f} GB")
    print(f"Versión CUDA:     {torch.version.cuda}")

---
## 🔢 Sección 2 — Benchmark: Multiplicación de Matrices

La multiplicación de matrices es la operación central en redes neuronales (capas densas, atención).  
Vamos a medir cuánto tarda en CPU versus GPU para matrices de distintos tamaños.

### ¿Por qué matrices?
Una capa `Linear(d_in, d_out)` realiza: `Y = X @ W + b`  
Con `X` de forma `(batch, d_in)` y `W` de forma `(d_in, d_out)` → esto es una GEMM (*General Matrix Multiply*).

In [ ]:
import torch
import time
import numpy as np

def benchmark_matmul(size: int, repeticiones: int = 5, device: str = 'cpu') -> float:
    """
    Mide el tiempo promedio de multiplicación de matrices cuadradas de `size x size`.
    Devuelve el tiempo en segundos.
    """
    A = torch.randn(size, size, dtype=torch.float32, device=device)
    B = torch.randn(size, size, dtype=torch.float32, device=device)

    # Calentamiento (warm-up): los primeros kernels son más lentos
    _ = torch.matmul(A, B)
    if device == 'cuda':
        torch.cuda.synchronize()  # Esperar que GPU termine

    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        C = torch.matmul(A, B)
        if device == 'cuda':
            torch.cuda.synchronize()  # ⚠️ Necesario para medir tiempo real en GPU
        fin = time.perf_counter()
        tiempos.append(fin - inicio)

    return np.mean(tiempos)


# ─── Ejecutar benchmark en múltiples tamaños ────────────────────────────────
tamanios = [256, 512, 1024, 2048, 4096]
resultados = []

print(f"{'Tamaño':>8} | {'CPU (ms)':>10} | {'GPU (ms)':>10} | {'Speedup':>8}")
print("-" * 48)

for sz in tamanios:
    t_cpu = benchmark_matmul(sz, device='cpu')
    t_gpu = benchmark_matmul(sz, device='cuda') if gpu_disponible else None

    speedup = t_cpu / t_gpu if t_gpu else None
    resultados.append({'size': sz, 'cpu_ms': t_cpu*1000,
                       'gpu_ms': (t_gpu*1000 if t_gpu else None),
                       'speedup': speedup})

    gpu_str = f"{t_gpu*1000:>10.2f}" if t_gpu else "N/A".rjust(10)
    sp_str  = f"{speedup:>8.1f}x" if speedup else "N/A"
    print(f"{sz:>8} | {t_cpu*1000:>10.2f} | {gpu_str} | {sp_str}")

print()
print("📌 Nota: cuda.synchronize() es esencial para medir correctamente.")
print("   Sin ella, medirías solo el tiempo de encolar el kernel, no de ejecutarlo.")

---
## 📊 Sección 3 — Visualización del Speedup
Graficamos los resultados para ver cómo el speedup crece con el tamaño del problema.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ─── Estilo oscuro tipo HPC ──────────────────────────────────────────────────
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0A0E1A')

szs      = [r['size']    for r in resultados]
cpu_ms   = [r['cpu_ms']  for r in resultados]
gpu_ms   = [r['gpu_ms']  for r in resultados if r['gpu_ms'] is not None]
speedups = [r['speedup'] for r in resultados if r['speedup'] is not None]

# ─── Gráfica 1: Tiempos CPU vs GPU ──────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#0F1629')
ax1.plot(szs, cpu_ms, 'o-', color='#FF4D8B', linewidth=2, markersize=7, label='CPU')
if gpu_ms:
    ax1.plot(szs[:len(gpu_ms)], gpu_ms, 's-', color='#00D4FF', linewidth=2, markersize=7, label='GPU T4')
ax1.set_xlabel('Tamaño de matriz (N×N)', color='#8892A4')
ax1.set_ylabel('Tiempo promedio (ms)', color='#8892A4')
ax1.set_title('Tiempo de ejecución: CPU vs GPU', color='white', fontsize=13, pad=12)
ax1.legend(fontsize=11)
ax1.tick_params(colors='#8892A4')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.15)
for spine in ax1.spines.values():
    spine.set_edgecolor('#1A2640')

# ─── Gráfica 2: Speedup ─────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0F1629')
if speedups:
    bars = ax2.bar(range(len(speedups)), speedups,
                   color=['#9B6DFF' if s < 50 else '#00E5A0' for s in speedups],
                   edgecolor='#1A2640', linewidth=0.5)
    ax2.set_xticks(range(len(speedups)))
    ax2.set_xticklabels([str(szs[i]) for i in range(len(speedups))], color='#8892A4')
    for bar, sp in zip(bars, speedups):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{sp:.1f}×', ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
ax2.set_xlabel('Tamaño de matriz (N×N)', color='#8892A4')
ax2.set_ylabel('Speedup GPU / CPU', color='#8892A4')
ax2.set_title('Speedup de GPU sobre CPU', color='white', fontsize=13, pad=12)
ax2.tick_params(colors='#8892A4')
ax2.grid(True, axis='y', alpha=0.15)
for spine in ax2.spines.values():
    spine.set_edgecolor('#1A2640')

plt.tight_layout(pad=2.5)
plt.suptitle('AI-HPC Benchmark — Multiplicación de Matrices', color='#00D4FF',
             fontsize=15, fontweight='bold', y=1.03)
plt.savefig('benchmark_resultado.png', dpi=150, bbox_inches='tight',
            facecolor='#0A0E1A')
plt.show()
print("\n💡 ¿Qué observas? El speedup crece con el tamaño del problema.")
print("   Matrices pequeñas tienen overhead de transferencia CPU→GPU.")
print("   Matrices grandes aprovechan el paralelismo masivo de la GPU.")

---
## ⏱️ Sección 4 — Profiling con CUDA Events

`torch.cuda.synchronize()` funciona pero es impreciso.  
El método profesional usa **CUDA Events** — contadores dentro de la GPU que miden con precisión de microsegundos.

In [ ]:
def benchmark_cuda_events(size: int, repeticiones: int = 20) -> dict:
    """
    Mide tiempo GPU usando CUDA Events (método profesional).
    Más preciso que time.perf_counter() + synchronize().
    """
    if not torch.cuda.is_available():
        return None

    A = torch.randn(size, size, dtype=torch.float32, device='cuda')
    B = torch.randn(size, size, dtype=torch.float32, device='cuda')

    # Warm-up
    for _ in range(3):
        _ = torch.matmul(A, B)
    torch.cuda.synchronize()

    tiempos_ms = []
    for _ in range(repeticiones):
        inicio = torch.cuda.Event(enable_timing=True)
        fin    = torch.cuda.Event(enable_timing=True)

        inicio.record()          # Marcar inicio en la línea de tiempo GPU
        C = torch.matmul(A, B)
        fin.record()             # Marcar fin

        torch.cuda.synchronize() # Esperar que ambos eventos se completen
        tiempos_ms.append(inicio.elapsed_time(fin))  # Tiempo en ms con μs precisión

    tiempos_ms = np.array(tiempos_ms)

    # Calcular FLOPS teóricos: 2*N^3 operaciones para matmul N×N
    flops = 2 * (size ** 3)
    tflops = flops / (tiempos_ms.mean() * 1e-3) / 1e12

    return {
        'size': size,
        'mean_ms': tiempos_ms.mean(),
        'std_ms':  tiempos_ms.std(),
        'min_ms':  tiempos_ms.min(),
        'max_ms':  tiempos_ms.max(),
        'tflops':  tflops
    }


# ─── Profiling detallado para 4096×4096 ─────────────────────────────────────
print("🔬 Profiling con CUDA Events (20 repeticiones, tamaño 4096×4096)\n")
prof = benchmark_cuda_events(4096, repeticiones=20)

if prof:
    print(f"  Tiempo promedio:  {prof['mean_ms']:.3f} ms")
    print(f"  Desviación std:   {prof['std_ms']:.3f} ms")
    print(f"  Tiempo mínimo:    {prof['min_ms']:.3f} ms")
    print(f"  Tiempo máximo:    {prof['max_ms']:.3f} ms")
    print(f"  Rendimiento:      {prof['tflops']:.2f} TFLOPS")
    print()
    print(f"📊 La GPU T4 tiene ~65 TFLOPS teóricos (FP32).")
    print(f"   Eficiencia lograda: {prof['tflops']/65*100:.1f}% del pico teórico")
    print()
    print("💡 ¿Por qué no llegamos al 100%?")
    print("   → Latencia de acceso a memoria HBM")
    print("   → Overhead del scheduler CUDA")
    print("   → La operación puede ser memory-bound, no compute-bound")

---
## 📐 Sección 5 — Ley de Amdahl

La **Ley de Amdahl** modela el speedup máximo cuando una fracción `p` del código es paralelizable:

$$S = \frac{1}{(1-p) + \frac{p}{s}}$$

Donde:
- `p` = fracción del código que puede ejecutarse en paralelo (GPU)
- `s` = speedup de esa fracción paralela
- `S` = speedup total del sistema

### ¿Qué nos dice?
> Aunque la GPU sea infinitamente rápida, la parte serial (`1-p`) limita el speedup total.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

def amdahl(p: float, s: float) -> float:
    """Calcula el speedup según la Ley de Amdahl."""
    return 1 / ((1 - p) + p / s)

# ─── Graficar curvas de Amdahl ───────────────────────────────────────────────
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0A0E1A')

s_vals = np.logspace(0, 3, 500)  # speedup paralelo de 1x a 1000x
colores = ['#FF4D8B', '#FFB800', '#00E5A0', '#00D4FF', '#9B6DFF']
fracciones = [0.50, 0.75, 0.90, 0.95, 0.99]

ax1 = axes[0]
ax1.set_facecolor('#0F1629')
for p, color in zip(fracciones, colores):
    speedups_teoricos = [amdahl(p, s) for s in s_vals]
    ax1.plot(s_vals, speedups_teoricos, color=color, linewidth=2,
             label=f'p = {int(p*100)}% paralelo → máx {1/(1-p):.0f}×')

ax1.set_xscale('log')
ax1.set_xlabel('Speedup de la parte paralela (GPU)', color='#8892A4')
ax1.set_ylabel('Speedup total del sistema', color='#8892A4')
ax1.set_title('Ley de Amdahl — Límite teórico', color='white', fontsize=13, pad=12)
ax1.legend(fontsize=9, loc='upper left')
ax1.tick_params(colors='#8892A4')
ax1.grid(True, alpha=0.15)
for spine in ax1.spines.values():
    spine.set_edgecolor('#1A2640')

# ─── Gráfica 2: Nuestro experimento en la ley de Amdahl ─────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0F1629')

# Estimamos fracción paralela desde el benchmark (asumimos parte serial ~10%)
p_estimado = 0.95  # 95% paralelizable (ajustar según tus resultados)
speedup_gpu = speedups[-1] if speedups else 100

s_range = np.logspace(0, 3, 500)
ax2.plot(s_range, [amdahl(p_estimado, s) for s in s_range],
         color='#00D4FF', linewidth=2, label=f'p = {int(p_estimado*100)}%')

# Marcar nuestro punto medido
if speedups:
    sp_medido = speedup_gpu
    speedup_total_real = amdahl(p_estimado, sp_medido)
    ax2.scatter([sp_medido], [speedup_total_real], color='#FFB800', s=120, zorder=5,
                label=f'Tu GPU: {sp_medido:.1f}× → Total: {speedup_total_real:.1f}×')
    ax2.axhline(y=1/(1-p_estimado), color='#FF4D8B', linestyle='--', alpha=0.6,
                label=f'Límite Amdahl: {1/(1-p_estimado):.0f}×')

ax2.set_xscale('log')
ax2.set_xlabel('Speedup GPU (matmul)', color='#8892A4')
ax2.set_ylabel('Speedup total del sistema', color='#8892A4')
ax2.set_title('Tu benchmark vs Ley de Amdahl', color='white', fontsize=13, pad=12)
ax2.legend(fontsize=9)
ax2.tick_params(colors='#8892A4')
ax2.grid(True, alpha=0.15)
for spine in ax2.spines.values():
    spine.set_edgecolor('#1A2640')

plt.tight_layout(pad=2.5)
plt.suptitle('Ley de Amdahl — AI-HPC Taller', color='#00D4FF',
             fontsize=15, fontweight='bold', y=1.03)
plt.savefig('amdahl_resultado.png', dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
plt.show()

---
## 🧠 Sección 6 — Inferencia con un Modelo Real (gratuito)

Ahora veremos un caso práctico de **AI + HPC**: inferencia de una red neuronal en GPU.  
Usamos **ResNet-18** pre-entrenado (incluido en PyTorch) — sin descargar nada externo.

In [ ]:
import torchvision.models as models
import torch

# ─── Cargar modelo pre-entrenado ─────────────────────────────────────────────
print("⏳ Cargando ResNet-18 pre-entrenado...")
modelo = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
modelo.eval()
print(f"✅ Modelo cargado: {sum(p.numel() for p in modelo.parameters()):,} parámetros")

# Batch de imágenes sintéticas (224×224, como ImageNet)
batch_sizes = [1, 8, 32, 128]
repeticiones = 20

print(f"\n{'Batch':>6} | {'CPU (ms/img)':>13} | {'GPU (ms/img)':>13} | {'Speedup':>9}")
print("-" * 52)

resultados_inf = []

for bs in batch_sizes:
    imagenes = torch.randn(bs, 3, 224, 224)

    # ── CPU ──
    modelo_cpu = modelo
    with torch.no_grad():
        _ = modelo_cpu(imagenes)  # warm-up
    tiempos_cpu = []
    with torch.no_grad():
        for _ in range(repeticiones):
            t0 = time.perf_counter()
            _ = modelo_cpu(imagenes)
            tiempos_cpu.append((time.perf_counter() - t0) * 1000)
    t_cpu_img = np.mean(tiempos_cpu) / bs

    # ── GPU ──
    t_gpu_img = None
    if torch.cuda.is_available():
        modelo_gpu = modelo.cuda()
        imagenes_gpu = imagenes.cuda()
        with torch.no_grad():
            _ = modelo_gpu(imagenes_gpu)  # warm-up
        torch.cuda.synchronize()
        tiempos_gpu = []
        with torch.no_grad():
            for _ in range(repeticiones):
                inicio = torch.cuda.Event(enable_timing=True)
                fin    = torch.cuda.Event(enable_timing=True)
                inicio.record()
                _ = modelo_gpu(imagenes_gpu)
                fin.record()
                torch.cuda.synchronize()
                tiempos_gpu.append(inicio.elapsed_time(fin))
        t_gpu_img = np.mean(tiempos_gpu) / bs

    sp = t_cpu_img / t_gpu_img if t_gpu_img else None
    resultados_inf.append({'bs': bs, 'cpu': t_cpu_img, 'gpu': t_gpu_img, 'sp': sp})

    gpu_s = f"{t_gpu_img:>13.3f}" if t_gpu_img else "N/A".rjust(13)
    sp_s  = f"{sp:>9.1f}×" if sp else "N/A"
    print(f"{bs:>6} | {t_cpu_img:>13.3f} | {gpu_s} | {sp_s}")

print()
print("💡 Observa: el speedup crece con el batch size.")
print("   Batches más grandes aprovechan mejor el paralelismo de la GPU.")

---
## 📊 Sección 7 — Throughput: Imágenes por segundo
La métrica real en producción no es la latencia por imagen, sino el **throughput** (imágenes/segundo).

In [ ]:
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0A0E1A')

bsizes = [r['bs']  for r in resultados_inf]
tp_cpu = [1000/r['cpu'] for r in resultados_inf]  # imágenes/seg
tp_gpu = [1000/r['gpu'] if r['gpu'] else None for r in resultados_inf]
sps    = [r['sp'] for r in resultados_inf]

x = np.arange(len(bsizes))
w = 0.35

# ─── Throughput ─────────────────────────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#0F1629')
ax1.bar(x - w/2, tp_cpu, w, color='#FF4D8B', label='CPU', edgecolor='#0A0E1A')
if any(tp_gpu):
    ax1.bar(x + w/2, [t for t in tp_gpu if t], w, color='#00D4FF', label='GPU T4', edgecolor='#0A0E1A')
ax1.set_xticks(x)
ax1.set_xticklabels([f'batch={b}' for b in bsizes], color='#8892A4')
ax1.set_ylabel('Imágenes / segundo', color='#8892A4')
ax1.set_title('Throughput — ResNet-18 Inferencia', color='white', fontsize=12, pad=10)
ax1.legend(fontsize=10)
ax1.tick_params(colors='#8892A4')
ax1.grid(True, axis='y', alpha=0.15)
for spine in ax1.spines.values():
    spine.set_edgecolor('#1A2640')

# ─── Speedup por batch size ──────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0F1629')
valid_sp = [(b, s) for b, s in zip(bsizes, sps) if s is not None]
if valid_sp:
    bs_v, sp_v = zip(*valid_sp)
    bars = ax2.bar(range(len(bs_v)), sp_v,
                   color=['#9B6DFF' if s < 15 else '#00E5A0' for s in sp_v],
                   edgecolor='#0A0E1A')
    for bar, sp in zip(bars, sp_v):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{sp:.1f}×', ha='center', va='bottom', color='white',
                 fontsize=11, fontweight='bold')
    ax2.set_xticks(range(len(bs_v)))
    ax2.set_xticklabels([f'batch={b}' for b in bs_v], color='#8892A4')
ax2.set_ylabel('Speedup GPU/CPU', color='#8892A4')
ax2.set_title('Speedup por Batch Size', color='white', fontsize=12, pad=10)
ax2.tick_params(colors='#8892A4')
ax2.grid(True, axis='y', alpha=0.15)
for spine in ax2.spines.values():
    spine.set_edgecolor('#1A2640')

plt.tight_layout(pad=2.5)
plt.suptitle('Inferencia ResNet-18 — CPU vs GPU T4', color='#00D4FF',
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig('throughput_resultado.png', dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
plt.show()

---
## 🏆 Sección 8 — Reporte Final del Taller
Generamos un resumen estructurado con todos los hallazgos.

In [ ]:
from IPython.display import display, HTML
import datetime

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
max_sp_matmul = max(speedups) if speedups else 0
max_sp_inf    = max([r['sp'] for r in resultados_inf if r['sp']], default=0)
limite_amdahl_95 = 1 / (1 - 0.95)

html = f"""
<div style="background:#0F1629; border:1px solid #00D4FF; border-radius:12px;
            padding:24px; font-family:monospace; color:#C5CDD8; max-width:700px">
  <h2 style="color:#00D4FF; margin-top:0">📋 Reporte — AI-HPC Taller</h2>
  <p style="color:#8892A4">Generado: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}</p>
  <hr style="border-color:#1A2640">

  <h3 style="color:#FFB800">Hardware</h3>
  <ul>
    <li>GPU: <b style="color:#00D4FF">{gpu_name}</b></li>
    <li>Entorno: Google Colab (gratuito)</li>
  </ul>

  <h3 style="color:#FFB800">Benchmark: Multiplicación de Matrices</h3>
  <ul>
    <li>Speedup máximo (4096×4096): <b style="color:#00E5A0">{max_sp_matmul:.1f}×</b></li>
    <li>Comportamiento: el speedup <b>crece</b> con el tamaño del problema</li>
    <li>Matrices pequeñas: dominadas por overhead de transferencia CPU→GPU</li>
  </ul>

  <h3 style="color:#FFB800">Benchmark: Inferencia ResNet-18</h3>
  <ul>
    <li>Speedup máximo (batch=128): <b style="color:#00E5A0">{max_sp_inf:.1f}×</b></li>
    <li>Throughput GPU vs CPU: órdenes de magnitud superiores</li>
  </ul>

  <h3 style="color:#FFB800">Ley de Amdahl (p = 95%)</h3>
  <ul>
    <li>Speedup teórico máximo: <b style="color:#9B6DFF">{limite_amdahl_95:.0f}×</b></li>
    <li>⚠️ Aunque la GPU sea ∞ rápida, el 5% serial limita el sistema</li>
    <li>Conclusión: optimizar el código serial es tan crítico como la GPU</li>
  </ul>

  <hr style="border-color:#1A2640">
  <p style="color:#8892A4; font-size:0.85em">
    CINF102 · Fundamentos de Cómputo de Alto Desempeño · 2026-10
  </p>
</div>
"""

display(HTML(html))

---
## 🎯 Preguntas de Reflexión

Discute con tu pareja y anota tus respuestas:

1. **¿Por qué el speedup de la GPU aumenta con el tamaño de la matriz?**  
   *(Pista: latencia vs throughput, overhead de transferencia)*

2. **Si el 10% de tu código de IA no puede paralelizarse, ¿cuál es el speedup máximo posible?**  
   *(Aplica la fórmula de Amdahl: `S = 1 / (0.10 + 0.90/∞)`)*

3. **En un cluster de 100 GPUs, ¿qué parte del sistema sería el nuevo cuello de botella?**  
   *(Pista: mira la slide de Desafíos Críticos)*

4. **¿Cómo cambiaría el benchmark si usaras precisión FP16 en lugar de FP32?**  
   *(Prueba: cambia `dtype=torch.float16` en la Sección 2)*

---

## 🔭 Extensión Opcional — Prueba con FP16 (Mixed Precision)

```python
# Ejecuta esto y compara con los resultados de FP32:
A = torch.randn(4096, 4096, dtype=torch.float16, device='cuda')
B = torch.randn(4096, 4096, dtype=torch.float16, device='cuda')

inicio = torch.cuda.Event(enable_timing=True)
fin    = torch.cuda.Event(enable_timing=True)
inicio.record()
C = torch.matmul(A, B)
fin.record()
torch.cuda.synchronize()
print(f"FP16 tiempo: {inicio.elapsed_time(fin):.3f} ms")
# Las GPU T4 tienen Tensor Cores que aceleran FP16 ~2-4× más que FP32
```

---
**¡Taller completado!** 🎉  
Guardaste evidencia de que la GPU es entre 10× y 100× más rápida para operaciones de IA,  
y que la Ley de Amdahl establece el límite teórico del sistema completo.
